# End-to-End LLM Training Example

## Overview

Complete example of training a language model with distributed training techniques.

### Topics Covered
- Model setup
- Data loading
- Training loop
- Checkpointing

In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

class SimpleTransformer(nn.Module):
    """Simple transformer for demonstration."""
    def __init__(self, vocab_size=50000, d_model=512, nhead=8, num_layers=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return self.fc(x)

In [ ]:
def train_step(model, batch, optimizer, scaler=None):
    """Single training step with optional AMP."""
    optimizer.zero_grad()
    
    if scaler:  # Mixed precision
        with torch.cuda.amp.autocast():
            output = model(batch['input_ids'])
            loss = nn.functional.cross_entropy(
                output.view(-1, output.size(-1)),
                batch['labels'].view(-1)
            )
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        output = model(batch['input_ids'])
        loss = nn.functional.cross_entropy(
            output.view(-1, output.size(-1)),
            batch['labels'].view(-1)
        )
        loss.backward()
        optimizer.step()
    
    return loss.item()

## Training Configuration

```python
# Launch with torchrun
# torchrun --nproc_per_node=4 train.py

config = {
    'batch_size': 32,
    'learning_rate': 1e-4,
    'num_epochs': 10,
    'gradient_accumulation': 4,
    'mixed_precision': True,
    'checkpoint_interval': 1000,
}
```